# DWN on JSC — training notebook (Kaggle)

Phase 1b of the dwn-fpga project. Produces a trained n=6 DWN checkpoint on JSC for the
exporter to consume.

**Why this runs on Kaggle and not locally:** upstream `torch_dwn` has no CPU path —
`lut_layer.py` raises `EFDFunction CPU not Implemented` in both forward and backward, and only
a CUDA extension ships. See project-brief.md §12 risk #7.

## Before you run anything

In the Kaggle settings panel on the right:

1. **Accelerator → GPU** (P100 or T4 x2, either is fine)
2. **Internet → On** — *off by default*. Without it the git clone and the OpenML fetch both fail.

## What you get out

Two files in `/kaggle/working/`, both tagged with `RUN_NAME` (derived from the config, so runs
never overwrite each other):

- `dwn_jsc_<run>_checkpoint.pt` — model `state_dict`, the full config, **the thermometer
  thresholds**, and the scaler params. The last two are not in the `state_dict` but are absolutely
  part of the model — the hardware encoder is undefined without them.
- `dwn_jsc_<run>_testvectors.npz` — 1000 test samples and the model's predictions on them. This is
  what the **Gate 1** golden-model testbench checks the RTL against.

Download both and commit them under `training/artifacts/`.


In [ ]:
# ---- environment check: fail loudly and early ----
import subprocess, sys, torch

print('torch     :', torch.__version__)
print('cuda avail:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu       :', torch.cuda.get_device_name(0))
    print('cuda (torch built against):', torch.version.cuda)
else:
    raise SystemExit('No GPU. Set Accelerator -> GPU in the settings panel. '
                     'DWN training cannot run on CPU.')

print()
print(subprocess.run(['nvcc', '--version'], capture_output=True, text=True).stdout)
print('If the nvcc CUDA version and the torch CUDA version differ a lot, the extension')
print('build in the next cell is where it will show up.')

In [ ]:
# ---- clone upstream DWN at the pinned commit and build the CUDA extension ----
# Pin matches third_party/DWN in the repo. Do not float this to main: the exporter is
# built against whatever checkpoint format this commit produces (CLAUDE.md).
PINNED_COMMIT = '9f887a0b4bd84dabf6d8c9ae35368ab2a7e0e3c0'

!rm -rf /kaggle/working/DWN
!git clone --quiet https://github.com/alanbacellar/DWN.git /kaggle/working/DWN
!cd /kaggle/working/DWN && git checkout --quiet {PINNED_COMMIT} && git log -1 --format='pinned at %h %ad %s'

# Confirm the CUDA sources actually exist at this pin before spending 5 minutes on a build.
# The pinned commit is literally "Delete custom_operators/cuda directory (Duplicate)" -- it
# removed a duplicate copy, not the real one, but that is worth verifying rather than assuming.
!ls -la /kaggle/working/DWN/src/torch_dwn/custom_operators/cuda/

# This compiles efd_cuda_kernel.cu with nvcc. Expect 2-5 minutes. It is the slowest and
# most fragile step in the notebook.
#
# --no-build-isolation is REQUIRED, not an optimization. Upstream's pyproject.toml declares
#     [build-system] requires = ["setuptools>=42", "wheel", "torch"]
# so a plain `pip install .` builds in a fresh isolated env and downloads ANOTHER torch from
# PyPI. setup.py's `import torch` then resolves to that one instead of the session's, so the
# extension gets built against a torch/CUDA pair that does not match the runtime -- it either
# fails to compile outright or builds and then fails to import on an ABI mismatch.
#
# Full output on purpose. Do NOT pipe this through `tail`: real compiler errors appear near
# the TOP of the log, while the last 20 lines are always the same generic pip epilogue
# ("did not run successfully / See above for output"), which identifies nothing.
!cd /kaggle/working/DWN && pip install --no-build-isolation .


In [ ]:
# ---- VERIFY the extension actually built ----
# This cell exists because the failure is otherwise silent. lut_layer.py does
#     if torch.cuda.is_available(): import efd_cuda
# so a failed build produces no error at install time -- you would instead get a bare
# NameError at the first forward pass, long after the real cause.
import torch, torch_dwn as dwn

try:
    import efd_cuda
    print('efd_cuda imported OK')
except ImportError as e:
    raise SystemExit(
        'efd_cuda failed to import -- the CUDA extension did not build.\n'
        'Scroll to the TOP of the install cell output and read the first compiler error;\n'
        'the tail of a pip failure is generic boilerplate and never names the cause.\n'
        f'Original error: {e}'
    )

# tiny end-to-end forward+backward, so we find out here rather than 200 lines later
_probe = torch.nn.Sequential(dwn.LUTLayer(12, 6, n=6), dwn.GroupSum(k=2, tau=1.0)).cuda()
_x = (torch.rand(4, 12, device='cuda') > 0.5).float()
_out = _probe(_x)
_out.sum().backward()
print('forward + backward OK, output shape', tuple(_out.shape))
del _probe, _x, _out


In [ ]:
# ---- CONFIG ----
# One dict drives the whole flow. This mirrors the Phase 2 requirement in docs/dse-plan.md §1:
# a sweep point is a config, not a code edit. Keep it that way.
#
# THIS RUN IS A REPRODUCTION ATTEMPT, not an exploration. Every value below is the paper's
# JSC "sm" (1x 50) configuration, from Table 14 of arXiv:2410.11112. Target: 74.0%.
# See docs/paper-configs.md. Don't tune anything here until 74% is reproduced -- the whole
# point is a known-good reference so Gate 1b can attribute any later hardware/software
# disagreement to the RTL rather than to the model.
CONFIG = {
    # --- encoding (Group A) ---
    'thermometer': 'distributive',   # paper: Distributive Thermometer, all datasets
    # z=200. The paper uses 200 for EVERY JSC config (sm/md/lg alike) and never sweeps it.
    # 16 features x 200 = 3200 input bits. Our earlier t=4 and t=8 runs were 25-50x below
    # the paper's operating point, which is why they plateaued at ~72.5%: with a single
    # learnable-mapped layer the thermometer is a FEATURE POOL that Learnable Mapping
    # selects from, and 50 nodes x n=6 = 300 slots choosing from 128 candidates is starved.
    'thermometer_bits': 200,
### MEMORY NOTE: 3200 input bits makes the binarized train set 8.5 GB as float32.
### Cells 6 and 8 now chunk and keep it in uint8 on the CPU. Do not "simplify" them
### back to a single .cuda() call -- that is a guaranteed OOM on a T4.

    # --- architecture (Group A) ---
    'n': 6,                          # paper: n=6 for all JSC models
    # SINGLE layer. Every JSC model in Table 14 is "1x N" -- not one uses a second layer.
    # Our old [300, 100] put a RANDOM-wired second layer between the encoder and the
    # popcount, which discarded most of layer 0's output. 50 nodes on a rich input beats
    # 400 nodes on a starved one: 74.0% vs our 72.6%.
    'layers': [50],
    # Learnable Mapping on the only layer. With z=200 this is what makes the model work --
    # it picks 300 useful thresholds out of 3200 candidates, at zero inference cost.
    'mapping': ['learnable'],

    # --- reduction ---
    'num_classes': 5,
    # Paper's tau for the 1x 50 config specifically. tau tracks layer width across their
    # JSC configs (1/0.7, 1/0.3, 1/0.1, 1/0.03 for 10, 50, 360, 2400 nodes) -- it rescales
    # the popcount as group size grows. If 'layers' changes, this should too.
    'tau': 1 / 0.3,

    # --- training (does not affect hardware) ---
    # Paper: BS=100, LR 1e-2(14), 1e-3(14), 1e-4(4) = 32 epochs total.
    # StepLR(step=14, gamma=0.1) over 32 epochs reproduces that schedule exactly.
    # (Table 14's caption claims 100 epochs for all models, but the JSC rows sum to 32.
    # The caption is wrong for JSC; the per-row schedule is authoritative.)
    'batch_size': 100,
    'epochs': 32,
    'lr': 1e-2,
    'lr_step': 14,
    'lr_gamma': 0.1,
    'seed': 20260802,
}

assert len(CONFIG['mapping']) == len(CONFIG['layers']), \
    "CONFIG['mapping'] needs exactly one entry per layer in CONFIG['layers']"
assert CONFIG['layers'][-1] % CONFIG['num_classes'] == 0, \
    ('final layer width must be divisible by num_classes -- GroupSum zero-pads silently '
     'otherwise, see docs/checkpoint-format.md §4')

RUN_NAME = 't{}_{}_{}_{}_b{}'.format(
    CONFIG['thermometer_bits'],
    CONFIG['thermometer'],
    '-'.join(str(w) for w in CONFIG['layers']),
    ''.join(m[0] for m in CONFIG['mapping']),   # 'l' = learnable, 'r' = random
    CONFIG['batch_size'],
)

# Predicted core area, from the docs/dse-plan.md §5 model: one node == one LUT6.
core_luts = sum(CONFIG['layers'])
input_bits = 16 * CONFIG['thermometer_bits']
selected_bits = core_luts * CONFIG['n']
print('run                :', RUN_NAME)
print('config:', CONFIG)
print()
print('input bits         :', input_bits)
print('predicted core LUTs:', core_luts, '(encoder NOT included)')
print('pct of xc7a35t     : {:.2f}%'.format(100 * core_luts / 20800))
print()
print('paper reference    : DWN (n=6; sm) 1x50 -> 74.0% accuracy, 110 LUTs total')
print('                     (Table 2, xcvu9p out-of-context. 110 total vs 50 core means')
print('                      ~60 LUTs of encoder + popcount on THEIR part.)')
print()
print(f'Of the {input_bits} thermometer bits, at most {selected_bits} are ever wired to')
print('anything -- Learnable Mapping selects output_size*n of them and the rest feed no')
print('node at all. So z=200 does NOT imply a 3200-bit encoder in hardware. How much those')
print(f'{selected_bits} surviving comparators actually cost on an Artix-7 is an open')
print('question for Phase 2, and it is one the paper never had to ask (brief §12 risk #3).')


In [ ]:
# ---- load JSC ----
# hls4ml_lhc_jets_hlf: 16 high-level jet-substructure features, 5 classes (g, q, w, z, t).
# This is the standard low-latency FPGA-ML benchmark -- see project-brief.md §7 and §8.
import numpy as np, torch
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder

torch.manual_seed(CONFIG['seed'])
np.random.seed(CONFIG['seed'])

data = fetch_openml('hls4ml_lhc_jets_hlf', version=1, as_frame=True)
X = data.data.to_numpy(dtype=np.float32)
y_raw = data.target.to_numpy()

label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y_raw)

print('X', X.shape, ' y', y.shape)
print('features:', list(data.feature_names))
print('classes :', list(label_encoder.classes_))
assert X.shape[1] == 16, f'expected 16 features, got {X.shape[1]}'
assert len(label_encoder.classes_) == CONFIG['num_classes']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=CONFIG['seed'], stratify=y)

# Standardize on train statistics only. The scaler is part of the model: the hardware
# encoder's thresholds live in this scaled space, so it gets saved with the checkpoint.
scaler = StandardScaler().fit(X_train)
X_train = scaler.transform(X_train).astype(np.float32)
X_test = scaler.transform(X_test).astype(np.float32)

X_train_t = torch.from_numpy(X_train)
X_test_t = torch.from_numpy(X_test)
y_train_t = torch.from_numpy(y_train).long()
y_test_t = torch.from_numpy(y_test).long()
print('train', X_train_t.shape, ' test', X_test_t.shape)

In [ ]:
# ---- thermometer binarization ----
# Pure PyTorch upstream, so this part would run fine locally too. Thresholds are fitted on
# TRAIN ONLY and must be saved -- Thermometer is not an nn.Module, so they are NOT in the
# state_dict, and without them the exported hardware encoder is undefined.
#
# CHUNKED ON PURPOSE. binarization.py does `(x.unsqueeze(-1) > thresholds).float()`, so a
# one-shot call at z=200 allocates 664000 x 16 x 200 x 4B = 8.5 GB in a single tensor and
# dies. We binarize in slices and store uint8 (2.1 GB), converting to float per batch on the
# GPU instead. Do not collapse this back into a single call.
import torch_dwn as dwn

THERMOMETERS = {
    'plain': dwn.Thermometer,
    'gaussian': dwn.GaussianThermometer,
    'distributive': dwn.DistributiveThermometer,
}

thermometer = THERMOMETERS[CONFIG['thermometer']](CONFIG['thermometer_bits']).fit(X_train_t)


def binarize_chunked(therm, x, chunk=10000):
    """Binarize -> flatten -> uint8, in slices, to bound peak memory."""
    out = torch.empty((x.size(0), x.size(1) * therm.num_bits), dtype=torch.uint8)
    for i in range(0, x.size(0), chunk):
        block = therm.binarize(x[i:i + chunk]).flatten(start_dim=1)
        out[i:i + chunk] = block.to(torch.uint8)
        del block
    return out


xb_train = binarize_chunked(thermometer, X_train_t)
xb_test = binarize_chunked(thermometer, X_test_t)

print('thresholds shape:', tuple(thermometer.thresholds.shape), '(features x bits)')
print('binarized train :', tuple(xb_train.shape), xb_train.dtype)
print('binarized test  :', tuple(xb_test.shape), xb_test.dtype)
print('input width     :', xb_train.size(1), 'bits')
print()
print('CPU memory held : {:.2f} GB as uint8 (would be {:.2f} GB as float32)'.format(
    (xb_train.numel() + xb_test.numel()) / 1e9,
    4 * (xb_train.numel() + xb_test.numel()) / 1e9))
print('These stay on the CPU. Batches are moved to the GPU and cast to float in the train')
print('loop -- one batch is {:.1f} MB, which is what makes z=200 fit at all.'.format(
    CONFIG['batch_size'] * xb_train.size(1) * 4 / 1e6))

assert xb_train.size(1) == 16 * CONFIG['thermometer_bits']
assert xb_train.dtype == torch.uint8
# Binarization must be exactly 0/1 -- the LUT address logic thresholds at > 0, so anything
# else would silently change addressing (docs/checkpoint-format.md §2).
assert xb_train[:1000].max() <= 1, 'binarized data must be 0/1 only'


In [ ]:
# ---- build the model ----
from torch import nn

layers, in_size = [], xb_train.size(1)
for i, width in enumerate(CONFIG['layers']):
    layers.append(dwn.LUTLayer(
        in_size, width, n=CONFIG['n'],
        mapping=CONFIG['mapping'][i],
    ))
    in_size = width
layers.append(dwn.GroupSum(k=CONFIG['num_classes'], tau=CONFIG['tau']))

model = nn.Sequential(*layers).cuda()
print(model)
print()
for i, m in enumerate(CONFIG['mapping']):
    print(f'layer {i}: {CONFIG["layers"][i]:4d} nodes, mapping={m}')
print()
for name, p in model.named_parameters():
    print(f'{name:30s} {str(tuple(p.shape)):20s} trainable={p.requires_grad}')


In [ ]:
# ---- train ----
# Data stays on the CPU as uint8; each batch is moved to the GPU and cast to float there.
# This is what upstream's examples/mnist.py does (`x_train[indices].cuda()`), and at z=200
# it is mandatory rather than stylistic -- preloading the full set is 8.5 GB of VRAM.
import time
from torch.nn.functional import cross_entropy

optimizer = torch.optim.Adam(model.parameters(), lr=CONFIG['lr'])
scheduler = torch.optim.lr_scheduler.StepLR(
    optimizer, step_size=CONFIG['lr_step'], gamma=CONFIG['lr_gamma'])


def evaluate(chunk=5000):
    """Chunked so the test set never lands on the GPU all at once (2.1 GB at z=200)."""
    model.eval()
    correct = 0
    with torch.no_grad():
        for i in range(0, xb_test.size(0), chunk):
            xb = xb_test[i:i + chunk].cuda().float()
            pred = model(xb).argmax(dim=1)
            correct += (pred == y_test_t[i:i + chunk].cuda()).sum().item()
    return correct / xb_test.size(0)


n_samples = xb_train.size(0)
best_acc, history, epoch_losses = 0.0, [], []
print(f'{n_samples // CONFIG["batch_size"]} optimizer steps per epoch '
      f'at batch_size={CONFIG["batch_size"]}')
t0 = time.time()

for epoch in range(CONFIG['epochs']):
    model.train()
    perm = torch.randperm(n_samples)          # CPU: indexes the CPU-resident uint8 tensor
    running, nb = 0.0, 0
    for i in range(0, n_samples, CONFIG['batch_size']):
        idx = perm[i:i + CONFIG['batch_size']]
        xb = xb_train[idx].cuda().float()
        yb = y_train_t[idx].cuda()
        optimizer.zero_grad()
        loss = cross_entropy(model(xb), yb)
        loss.backward()
        optimizer.step()
        running += loss.item()
        nb += 1
    scheduler.step()

    epoch_loss = running / nb
    epoch_losses.append(epoch_loss)
    acc = evaluate()
    best_acc = max(best_acc, acc)
    history.append(acc)
    print(f'epoch {epoch+1:3d}/{CONFIG["epochs"]}  mean loss {epoch_loss:.4f}  '
          f'test acc {acc:.4f}  [{time.time()-t0:.0f}s]')

print()
print(f'final {history[-1]:.4f}   best {best_acc:.4f}   {time.time()-t0:.0f}s')
print()
print('=== REPRODUCTION TARGET ===')
print('  paper DWN (n=6; sm) 1x50, z=200:  74.0%   (Table 2, arXiv:2410.11112)')
print(f'  this run:                         {100*best_acc:.2f}%')
print(f'  gap:                              {100*best_acc - 74.0:+.2f}pp')
print()
print('Our previous runs, all at z<=8 with a two-layer model:')
print('  t=4,   [300,100], b256:  72.31%')
print('  t=8,   [300,100], b256:  72.53%')
print('  t=8,   [300,100], b32:   72.60%')
print()
print('Within ~1pp of 74.0% means the training pipeline is sound and this checkpoint is the')
print('Gate 1 reference. Further off than that means something is still wrong, and it is NOT')
print('worth building an exporter against a model we cannot explain -- a wrong checkpoint')
print('exports faithfully into wrong hardware.')
print()
print(f'  loss  epoch 1 {epoch_losses[0]:.4f}  ->  min {min(epoch_losses):.4f}'
      f'  ->  final {epoch_losses[-1]:.4f}')
print(f'  acc   epoch 1 {100*history[0]:.2f}%  ->  best {100*best_acc:.2f}% at epoch '
      f'{1 + history.index(best_acc)}  ->  final {100*history[-1]:.2f}%')


In [ ]:
# ---- inspect what the exporter will actually have to read ----
# This is the checkpoint-format reconnaissance called for in project-brief.md §12 risk #5.
# Do not guess this format later -- record what you see here.
print('=== parameters ===')
for name, p in model.state_dict().items():
    print(f'{name:34s} {str(tuple(p.shape)):18s} {p.dtype}')

print()
print('=== LUT tables ===')
lut0 = model[0].luts
print('shape', tuple(lut0.shape), '= (output_size, 2**n)')
print('range [{:.3f}, {:.3f}] -- clamped to [-1,1] during training'.format(
    lut0.min().item(), lut0.max().item()))
print('ONLY THE SIGN MATTERS at inference: the exporter thresholds these at 0.')
print('first node, first 16 entries as bits:', (lut0[0][:16] > 0).int().tolist())

print()
print('=== mapping (which input bits each node reads) ===')
for i, layer in enumerate(model[:-1]):
    m = layer.mapping
    if isinstance(m, dwn.LearnableMapping):
        print(f'layer {i}: LearnableMapping -- take argmax over its weights to get fixed wiring')
        print('         weight shape', tuple(m.weights.shape))
    else:
        print(f'layer {i}: fixed tensor {tuple(m.shape)} {m.dtype} = (output_size, n)')
        print('         node 0 reads input bits', m[0].tolist())

In [ ]:
# ---- save the checkpoint ----
# Everything the exporter needs, in one file. The thermometer thresholds and the scaler are
# NOT in the state_dict but ARE part of the model -- the hardware encoder is undefined
# without them. At z=200 the thresholds tensor is (16, 200) and matters more than ever:
# the wiring selects which of those 3200 comparisons actually get built.
import numpy as np

CKPT = f'/kaggle/working/dwn_jsc_{RUN_NAME}_checkpoint.pt'
VECS = f'/kaggle/working/dwn_jsc_{RUN_NAME}_testvectors.npz'
FULL = f'/kaggle/working/dwn_jsc_{RUN_NAME}_testset_full.npz'

torch.save({
    'run_name': RUN_NAME,
    'config': CONFIG,
    'pinned_commit': PINNED_COMMIT,
    'state_dict': model.state_dict(),
    'thermometer': {
        'kind': CONFIG['thermometer'],
        'num_bits': CONFIG['thermometer_bits'],
        'thresholds': thermometer.thresholds.cpu(),
    },
    'scaler': {
        'mean': torch.from_numpy(scaler.mean_.astype(np.float32)),
        'scale': torch.from_numpy(scaler.scale_.astype(np.float32)),
    },
    # Alphabetical, straight from LabelEncoder: ['g','q','t','w','z']. This is NOT the
    # physics ordering used in project-brief.md §7 -- index 2 is top, not W. The 7-segment
    # display mapping and every comparison table have to use THIS order.
    'classes': list(label_encoder.classes_),
    'feature_names': list(data.feature_names),
    'results': {
        'final_acc': history[-1],
        'best_acc': best_acc,
        'history': history,
        'epoch_losses': epoch_losses,
    },
    'paper_target': 0.740,   # DWN (n=6; sm) 1x50, Table 2 of arXiv:2410.11112
    'torch_version': torch.__version__,
}, CKPT)

print('wrote', CKPT)

model.eval()

# ---- 1000-sample file: drives the RTL testbenches ----
# Both representations on purpose: x_raw exercises the design's own encoder end to end,
# x_binarized isolates the LUT core, so a Gate 1 failure can be attributed to one or the
# other rather than to "somewhere in the design".
with torch.no_grad():
    pred_1k = model(xb_test[:1000].cuda().float()).argmax(dim=1).cpu().numpy()

np.savez_compressed(
    VECS,
    x_binarized=xb_test[:1000].numpy(),      # already uint8
    x_raw=X_test[:1000],                     # scaled feature space, pre-thermometer
    y=y_test[:1000],
    pred=pred_1k,
)
print('wrote', VECS)
print(f'  x_binarized {tuple(xb_test[:1000].shape)} uint8  |  x_raw {X_test[:1000].shape}')

# ---- FULL test set: required for Gate 1b ----
# Gate 1b (project-brief.md §11) is not "it lights the right LED for a few inputs" -- the
# bitstream has to reproduce the software model's accuracy over the WHOLE test set, to the
# sample. 1000 samples cannot support that claim.
#
# x_binarized is deliberately NOT saved here. At 166k x 3200 bits it is ~530 MB, and nothing
# needs it: the board's own thermometer encoder turns features into bits, so the host only
# ever sends quantized features. The 1000-sample file above keeps x_binarized for the
# core-level testbench, which is its only consumer.
#
# `pred` is the SOFTWARE MODEL's output, not the ground truth, and that distinction is the
# point: Gate 1b asks whether hardware reproduces the model to the sample. Disagreement with
# `pred` is a hardware bug; disagreement with `y` is just the model being wrong, which is
# already known and measured.
preds_full = []
with torch.no_grad():
    for i in range(0, xb_test.size(0), 5000):
        preds_full.append(
            model(xb_test[i:i+5000].cuda().float()).argmax(dim=1).cpu().numpy())
preds_full = np.concatenate(preds_full)

np.savez_compressed(
    FULL,
    x_raw=X_test,            # (166000, 16) float32, scaled feature space
    y=y_test,
    pred=preds_full,
)
print('wrote', FULL)
print(f'  {X_test.shape[0]} samples, software accuracy '
      f'{100*(preds_full == y_test).mean():.2f}%')
print()
print('Download all three from the Output panel into training/artifacts/.')
print('The _testset_full.npz is LARGE and is gitignored -- it is fully regenerable by')
print('re-running this notebook, so it does not belong in git. Gate 1b needs it; the RTL')
print('testbenches do not.')


## After this runs

1. Download `dwn_jsc_<run>_checkpoint.pt` and `dwn_jsc_<run>_testvectors.npz` from the Output
   panel into `training/artifacts/`. Keep the previous run's files — the comparison *is* the
   result.
2. Add a row to the run log in `training/README.md`.
3. Record the accuracy against the §8 reference numbers, and read the **shape** of the curve, not
   just the final number. Flat from epoch 1 means something is capping the model, not that it
   needs more epochs.
4. Write the checkpoint structure from the inspection cell into the exporter's design notes —
   that closes project-brief.md §12 risk #5.

**Speeding up Phase 2:** the nvcc build is 2–5 minutes every fresh session. Before the sweep,
build the wheel once and save it as a Kaggle Dataset, then install from that instead of
recompiling. Not worth doing yet.
